In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from IPython.display import display
df_train = pd.read_csv(r'train_Titanic.csv')
df_test = pd.read_csv(r'test_Titanic.csv')
df_train.head()
df_train1 = df_train.drop(['Name'], axis=1)
df_train1['Sex1'] = df_train1['Sex'].map({'male': 0, 'female': 1})
df_train1['Sex2'] = df_train1['Sex'].map({'male': 1, 'female': 0})
# min_max_scaler = MinMaxScaler()
# min_max_scaler.fit(df_train1)
port_survival = df_train1.groupby('Embarked')['Survived'].agg(['mean', 'max', 'min'])
print(port_survival)
df_train1['Embarked_survived'] = df_train1.groupby('Embarked')['Survived'].transform('mean')
df_train1.head()
df_train1.drop(['Embarked'], axis=1, inplace=True)
df_train1['Fare_bin'] = pd.qcut(df_train1['Fare'], q=5, labels = [1 , 2,3,4,5],duplicates='drop')
df_train1['Fare_bin_survived'] = df_train1.groupby('Fare_bin', observed=True)['Survived'].transform('mean')
df_train1.head()

df_train1.drop(['Sex'], axis=1, inplace=True)
df_train1.head()
df_train1['Age'] = df_train1['Age'].fillna(df_train1['Age'].mean())
df_train1.isna().sum()
df_train1['Embarked_survived'] = df_train1['Embarked_survived'].fillna(df_train1['Embarked_survived'].mean())
MM = MinMaxScaler()
df_train1.head()
df_train1['Age'] = MM.fit_transform(df_train1[['Age']])
df_train1.head()
df_train1.drop(['Fare_bin' , 'PassengerId' , 'Fare'] , axis = 1 , inplace = True)
df_train1['Cabin_clean'] = df_train1['Cabin'].notna().astype(int)
df_train1.head()
ticket_counts = df_train1['Ticket'].value_counts()
df_train1['ticket_counts'] = df_train1['Ticket'].map(ticket_counts)
df_train1.head()
# print(df_train1.groupby('ticket_counts')['Survived'].mean()) 
##准备分箱 看到了票相同人数也有这个差异 分箱处理 回宿舍了 111
##把单片机和数电作业写了 就开始玩了 玩一会准备睡觉了
df_train1.head()
df_train1.drop(['Cabin'] , axis = 1 ,inplace = True)
df_train1.head()
#平滑处理
print(df_train1['ticket_counts'].value_counts())
print(df_train1.groupby('ticket_counts',observed= True)['Survived'].mean())
global_survived = df_train1['Survived'].mean()
m = 10
agg = df_train1.groupby('ticket_counts')['Survived'].agg(['count' , 'mean'])
counts = agg['count']
mean = agg['mean']
smooth = (counts * mean + m * global_survived) / (counts + m )
print(smooth)

In [74]:
# display(df_train1)
df_train1['ticket_counts_smooth'] = df_train1['ticket_counts'].map(smooth)
df_train1.head()
#对兄弟姐妹进行处理
df_train1.head()
df_train1.groupby('SibSp_counts')['Survived'].mean()
df_train1['SibSp_Survived'] = df_train1.groupby('SibSp')['Survived'].transform('mean')
summary = df_train1.groupby('SibSp')['Survived'].agg(['count' , 'mean'])
summary.columns = ['组内总人数', '生还率']
print(summary)
#对姐妹平滑处理
agg_SibSp = df_train1.groupby('SibSp')['Survived'].agg(['count' , 'mean'])
counts_SibSp = agg_SibSp['count']
means_SibSp = agg_SibSp['mean']
smooth_SibSp = (counts_SibSp * means_SibSp + m * global_survived) / (counts_SibSp + m)
df_train1['SibSp_Survived_smooth'] = df_train1['SibSp'].map(smooth_SibSp)
df_train1.head()
#如法炮制对Pclass
Pclass_Survived = df_train1.groupby('Pclass')['Survived'].transform('mean')
df_train1['Pclass_Survived'] = Pclass_Survived
df_train1.head()
#对Parch 除了平滑处理是否还有别的方式？
print(df_train1.groupby('Parch')['Survived'].mean())
print(df_train1['Parch'].value_counts())

       组内总人数       生还率
SibSp                 
0        608  0.345395
1        209  0.535885
2         28  0.464286
3         16  0.250000
4         18  0.166667
5          5  0.000000
8          7  0.000000
Parch
0    0.343658
1    0.550847
2    0.500000
3    0.600000
4    0.000000
5    0.200000
6    0.000000
Name: Survived, dtype: float64
Parch
0    678
1    118
2     80
5      5
3      5
4      4
6      1
Name: count, dtype: int64
